# CPC Domain-Aware Pretraining on Kaggle

**Setup:** Add two data sources in the Kaggle sidebar:
1. Dataset: `guozhenjennzhu/csi-bench` (the CSI-Bench data)
2. Your repo: upload as a Kaggle dataset or use `git clone`

**Runtime:** GPU T4 x2 → Settings → Accelerator → GPU

In [ ]:
# 1. Clone your repo
!git clone https://github.com/YOUR_USERNAME/csi-bench-SSL.git /kaggle/working/csi-bench-SSL
%cd /kaggle/working/csi-bench-SSL
!git checkout DANN  # your working branch

In [ ]:
# 2. Install deps (most are pre-installed on Kaggle)
!pip install -q pyaml tqdm scikit-learn seaborn h5py

In [ ]:
# 3. Symlink the Kaggle dataset into the expected data/ directory
import os
os.makedirs('data', exist_ok=True)

# The CSI-Bench dataset on Kaggle is at /kaggle/input/csi-bench/
# Check the actual structure:
!ls /kaggle/input/csi-bench/

In [ ]:
# Symlink each task directory into data/
# Adjust paths if the Kaggle dataset has a different structure
import glob

kaggle_data = '/kaggle/input/csi-bench'
tasks = ['BreathingDetection', 'FallDetection', 'Localization', 'MotionSourceRecognition',
         'HumanActivityRecognition', 'HumanIdentification', 'ProximityRecognition']

for task in tasks:
    src = os.path.join(kaggle_data, task)
    dst = os.path.join('data', task)
    if os.path.exists(src) and not os.path.exists(dst):
        os.symlink(src, dst)
        print(f'Linked {src} -> {dst}')
    elif not os.path.exists(src):
        # Maybe nested under a subdirectory
        candidates = glob.glob(f'/kaggle/input/csi-bench/*/{task}')
        if candidates:
            os.symlink(candidates[0], dst)
            print(f'Linked {candidates[0]} -> {dst}')
        else:
            print(f'WARNING: {task} not found in Kaggle dataset')

!ls -la data/

In [ ]:
# 4. Run domain-aware CPC pretraining with augmentations
# - GroupNorm + spectral norm in g_enc (architecture changes)
# - CSI augmentation (amp scaling, noise, subcarrier dropout, freq jitter)
# - Domain-aware hard negative ramp: 0.0 -> 0.8 over training
!python scripts/pretrain.py \
    --pretrain_method cpc_domain_aware \
    --all_tasks \
    --domain_neg_ratio 0.8 \
    --epochs 300 \
    --batch_size 32 \
    --learning_rate 3e-4 \
    --save_dir pretrain_results

In [ ]:
# 5. Check the training curve
import pandas as pd
import matplotlib.pyplot as plt

results_dirs = !ls -d pretrain_results/all_tasks/cpc/*/
latest = sorted(results_dirs)[-1]
print(f'Results dir: {latest}')

df = pd.read_csv(f'{latest}/training_results.csv')
plt.figure(figsize=(10, 5))
plt.plot(df['Epoch'], df['Train Loss'])
plt.xlabel('Epoch')
plt.ylabel('Train Loss')
plt.title('CPC Domain-Aware Pretraining (with augmentation)')
plt.grid(True)
plt.show()

print(f"Final loss: {df['Train Loss'].iloc[-1]:.4f}")
print(f"Best loss: {df['Train Loss'].min():.4f} at epoch {df.loc[df['Train Loss'].idxmin(), 'Epoch']}")

In [ ]:
# 6. Save the encoder weights — download this from Kaggle output
import shutil
encoder_path = f'{latest}/encoder_weights.pt'
output_path = '/kaggle/working/encoder_weights_domain_aware_aug.pt'
shutil.copy(encoder_path, output_path)
print(f'Encoder saved to {output_path}')
print('Download this file from the Output tab on the right sidebar.')

In [ ]:
# 7. (Optional) Also run standard CPC pretraining for comparison
!python scripts/pretrain.py \
    --pretrain_method cpc \
    --all_tasks \
    --epochs 300 \
    --batch_size 32 \
    --learning_rate 3e-4 \
    --save_dir pretrain_results